In [2]:
!pip install transformers

In [3]:
%pip install accelerate

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.6/138.6 MB 10.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 10.1 MB/s eta 0:00:0000:01
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 5.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install pydantic


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import json
import re
from typing import Optional
from pydantic import BaseModel, Field, ValidationError


# Define your expected output schema (customize fields to your task)
class ExtractionSchema(BaseModel):
  date: Optional[str] = Field(
      default=None, description="Extracted date or null"
  )
  location: Optional[str] = Field(
      default=None, description="Extracted location or null"
  )
  summary: Optional[str] = Field(
      default=None, description="Brief summary or null"
  )


def parse_and_validate_output(raw_output: str) -> dict:
  """Safely extracts JSON from raw LLM output and validates it against Pydantic."""
  json_str = None

  # 1. Try direct parsing
  try:
    data = json.loads(raw_output)
    if isinstance(data, dict):
      json_str = data
  except json.JSONDecodeError:
    pass

  # 2. Fallback: Search for JSON block via regex if direct parsing failed
  if not json_str:
    match = re.search(r"(\{.*?\})", raw_output, re.DOTALL)
    if match:
      try:
        data = json.loads(match.group(1))
        if isinstance(data, dict):
          json_str = data
      except json.JSONDecodeError:
        pass

  # 3. If we found a dictionary, validate with Pydantic
  if json_str:
    try:
      validated = ExtractionSchema(**json_str)
      return validated.dict(), True  # Success
    except ValidationError:
      # Schema mismatch, but we got JSON
      return json_str, False

  # 4. Total failure fallback
  return {"date": None, "location": None, "summary": None}, False

# 07 - Metadata extraction with EuroLLM

Run from the `notebooks/` folder. Install once: `%pip install transformers accelerate torch`.

Steps: (1) make a gold-standard template you verify by hand, (2) read the first pages of each PDF, (3) ask EuroLLM for JSON, (4) check the answers against the page text and against your gold rows.

Everything runs locally. Outputs go to `../output/extraction/`.

## 1. Settings

In [8]:
from pathlib import Path
import json, re, statistics, unicodedata, difflib
import pandas as pd
import pymupdf
from IPython.display import display

PDF_DIR = Path('../pdfs').resolve()
INVENTORY_DIR = Path('../output/inventory').resolve()
OUT_DIR = Path('../output/extraction').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)


MODEL_ID = MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
RUN_SCOPE = 'dev'        
FIRST_PAGES = 3              
MAX_INPUT_TOKENS = 3000      
FIELDS = ['title', 'authors', 'doi', 'publication_year', 'document_type', 'language', 'keywords']
LIST_FIELDS = {'authors', 'keywords'}
DOC_TYPES = ['journal_article', 'report', 'thesis', 'conference_paper', 'book_chapter', 'policy_brief', 'other']

read = lambda name: pd.read_csv(INVENTORY_DIR / name, dtype=str, keep_default_na=False, encoding='utf-8-sig')
documents, observations = read('documents.csv'), read('metadata_observations.csv')
print(len(documents), 'documents')

94 documents


## 2. Gold-standard template

Creates `gold_metadata.csv` once (never overwrites your edits): 40 documents, 25 `dev` and 15 `test`, pre-filled with the PDF-embedded candidates. **Verify every value against the PDF or the Publinova record.**

Fill-in rules: `;` between authors/keywords · `publication_year` as 4 digits · `document_type` from `journal_article, report, thesis, conference_paper, book_chapter, policy_brief, other` · `language` is `en`, `nl` or `mixed` · empty = not checked · `NONE` = checked and absent. Don't tune on `test` until the end.

In [10]:
GOLD_PATH = OUT_DIR / 'gold_metadata.csv'
if GOLD_PATH.exists():
    print('gold_metadata.csv exists; left untouched.')
else:
    pool = documents[documents.text_status.isin(['extractable', 'partial'])]
    pick = pool.sample(n=min(40, len(pool)), random_state=42)
    cand = lambda d, f: ' | '.join(observations[(observations.document_id == d) & (observations.field_name == f)].observed_value.unique())
    gold = pd.DataFrame({
        'document_id': pick.document_id, 'file_name': pick.file_name,
        'split': ['dev'] * min(25, len(pick)) + ['test'] * max(len(pick) - 25, 0),
        'title': [cand(d, 'title') for d in pick.document_id],
        'authors': [cand(d, 'author') for d in pick.document_id],
        'doi': [cand(d, 'doi') for d in pick.document_id],
        'publication_year': '', 'document_type': '', 'language': '',
        'keywords': [cand(d, 'pdf_keywords') for d in pick.document_id]})
    gold.to_csv(GOLD_PATH, index=False, encoding='utf-8-sig')
    print('Created', GOLD_PATH, '- now verify it by hand.')

Created /Users/georgyfurs/Documents/ICT_Year4/CPS/DDD/output/extraction/gold_metadata.csv - now verify it by hand.


## 3. Input and prompt

Each line of the first pages is read with its font size. Lines clearly larger than the page's typical text get a `[BIG]` marker, which helps the model find the title. The prompt ends with `{` so the model continues straight into a JSON object.

In [12]:
def page_text(page):
    lines = []
    for block in page.get_text('dict', sort=True)['blocks']:
        if block.get('type') != 0:
            continue
        for line in block['lines']:
            spans = [s for s in line['spans'] if s['text'].strip()]
            if spans:
                lines.append((max(s['size'] for s in spans), ''.join(s['text'] for s in spans).strip()))
    if not lines:
        return ''
    typical = statistics.median(size for size, _ in lines)
    return '\n'.join(('[BIG] ' if size >= 1.2 * typical else '') + text for size, text in lines)

def first_relevant_pages(pdf_path, min_chars=1200, max_pages=FIRST_PAGES):
    with pymupdf.open(PDF_DIR / pdf_path) as pdf:
        texts = []
        total = 0
        for i in range(min(max_pages + 1, len(pdf))):
            t = page_text(pdf[i])
            texts.append(f'--- page {i + 1} ---\n{t}')
            total += len(t.strip())
            if total >= min_chars:
                break
        return '\n\n'.join(texts)

SYSTEM = (
    'You are a bibliographic metadata extractor. '
    'You read the first page of a document (English or Dutch) and return its metadata as JSON. '
    'Use ONLY information printed in the document. If a field is not printed, use null or []. '
    'Lines starting with [BIG] are in a larger font and are usually the title. '
    'Respond with ONE JSON object and NOTHING else. '
    'Do not repeat these instructions. Do not add any text before or after the JSON. '
    'The JSON must have exactly these keys: '
    'title (string or null), '
    'authors (list of strings), '
    'doi (string or null, without "https://doi.org/" prefix), '
    'publication_year (integer or null, from "Published" or the journal issue year), '
    'document_type (one of: ' + ', '.join(DOC_TYPES) + '), '
    'language ("en", "nl", "mixed", or null), '
    'keywords (list of strings). '
    'Example output for a made-up document:\n'
    '{"title": "The Example Study", "authors": ["A. Smith", "B. Jones"], '
    '"doi": "10.1234/example", "publication_year": 2024, '
    '"document_type": "journal_article", "language": "en", '
    '"keywords": ["alpha", "beta"]}'
)

def user_prompt(text):
    return (
        'Extract the metadata from the document below and return it as a single JSON object.\n\n'
        'DOCUMENT:\n'
        f'{text}'
    )

demo = documents[documents.text_status != 'no_text'].iloc[0]
print(demo.file_name); print(first_relevant_pages(demo.pdf_path)[:1500])

Animals on the leeve-checkpoint.pdf
--- page 1 ---
[BIG] POLDER2C’s
Animals on the levee
Vana Tsimopoulou
U-Talent programma en de Hoogwater-excursie, 30-05-2022
ThisprojecthasreceivedfundingfromtheInterreg2Seasprogramme2014-2020co-fundedbythe European Regional Development Fund under subsidy contract No [2S07-023]

--- page 2 ---
[BIG] Flood protection vs Nature
A ‘conflict of interest’
story
Lessons from Polder2C’s
ThisprojecthasreceivedfundingfromtheInterreg2Seasprogramme2014-2020co-fundedbythe European Regional Development Fund under subsidy contract No [2S07-023]

--- page 3 ---
[BIG] The problem
Levee design
Engineers design levees without taking into account animal activity
(Source: Cobos Roa, 2015)
ThisprojecthasreceivedfundingfromtheInterreg2Seasprogramme2014-2020co-fundedbythe European Regional Development Fund under subsidy contract No [2S07-023]

--- page 4 ---
[BIG] The problem
…but in reality
•Levees attract burrowing animals
•Many of them are protected (e.g. beavers)
•It 

## 4. Load EuroLLM and run

In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
model = model.to("cpu")
model.eval()

def extract(text):
    messages = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user_prompt(text)}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(
    prompt,
    return_tensors='pt',
    truncation=True,
    max_length=MAX_INPUT_TOKENS,
    add_special_tokens=False,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id,
        )
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1104 20:21:18.323000 67856 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
RAW_PATH = OUT_DIR / 'eurollm_raw.jsonl'
runnable = documents[documents.text_status.isin(['extractable', 'partial'])]
if RUN_SCOPE == 'sample5':
    todo = runnable.head(5)
elif RUN_SCOPE == 'all':
    todo = runnable
else:
    g = pd.read_csv(GOLD_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    todo = runnable[runnable.document_id.isin(g.loc[g.split == RUN_SCOPE, 'document_id'])]

done = {}
if RAW_PATH.exists():
    for line in RAW_PATH.read_text(encoding='utf-8').splitlines():
        r = json.loads(line); done[(r['document_id'], r['model'])] = r['raw']

with RAW_PATH.open('a', encoding='utf-8') as sink:
    for n, row in enumerate(todo.itertuples(index=False), start=1):
        if (row.document_id, MODEL_ID) in done:
            continue
        raw = extract(first_relevant_pages(row.pdf_path))
        done[(row.document_id, MODEL_ID)] = raw
        sink.write(json.dumps({'document_id': row.document_id, 'model': MODEL_ID, 'raw': raw}, ensure_ascii=False) + '\n')
        sink.flush()
        print(f'[{n}/{len(todo)}] {row.file_name[:70]}')

[1/25] Animals on the leeve-checkpoint.pdf
[2/25] 002_alumni-loyalty-drivers-in-higher-education.pdf
[3/25] 008_effect-of-salinity-and-ph-on-growth-phycoerythrin-and-nonvolatile-
[4/25] 010_extraction-and-analysis-of-free-amino-acids-and-5-nucleotides-the-
[5/25] 013_grip-op-teksten-grip-op-kennis.pdf
[6/25] 016_impacts-of-counting-protocols-for-light-microscopy-on-estimates-of
[7/25] 020_long-term-forecast-of-thermal-mortality-with-climate-warming-in-ri
[8/25] 024_seasonal-variation-in-the-balance-and-strength-of-cooperative-and-
[9/25] 027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-agent-in-a-veget
[10/25] 029_transition-towards-integrated-flood-risk-management-strategies.pdf
[11/25] 032_what-is-in-a-student-faculty-relationship-a-template-analysis-of-s
[12/25] 034_animal-burrows-in-your-levees.pdf
[13/25] 038_accelerating-the-adoption-of-the-circular-economy-in-the-tourism-a
[14/25] 039_autonomous-shipping-technology-implications-in-logistics-chains-an
[15/25] 043_beweegreisk

## 5. Parse and sanity-check

Parsing never crashes on bad JSON; it records `parse_ok = False`. Titles, authors and DOIs are also checked against the page text: `grounded_* = False` means the model wrote something that is not actually on the first pages, which usually means a hallucination.

In [ ]:
def squash(s):
    s = unicodedata.normalize('NFKD', str(s)).lower()
    return re.sub(r'[^a-z0-9]+', '', ''.join(c for c in s if not unicodedata.combining(c)))

def clean_doi(v):
    return re.sub(r'(?i)^(https?://(dx\.)?doi\.org/|doi:\s*)', '', str(v).strip()).rstrip('.,;)').lower()

def as_list(v):
    if v is None or v == '': return []
    if isinstance(v, str): v = re.split(r'[;|]', v)
    return [str(x).strip() for x in v if str(x).strip()]

rows = []
for row in todo.itertuples(index=False):
    raw = done.get((row.document_id, MODEL_ID))
    if raw is None:
        continue
    rec = {'document_id': row.document_id, 'file_name': row.file_name, 'parse_ok': True}
    try:
        data, _ = json.JSONDecoder().raw_decode(raw)
    except Exception:
        data, rec['parse_ok'] = {}, False
    source = squash(first_relevant_pages(row.pdf_path))
    for f in FIELDS:
        v = data.get(f)
        if f in LIST_FIELDS:
            rec[f] = '; '.join(as_list(v))
        else:
            rec[f] = '' if v is None else (clean_doi(v) if f == 'doi' else str(v).strip())
    rec['grounded_title'] = bool(rec['title']) and squash(rec['title']) in source
    rec['grounded_doi'] = bool(rec['doi']) and squash(rec['doi']) in source
    names = as_list(rec['authors'])
    rec['grounded_authors'] = bool(names) and all(all(squash(t) in source for t in re.findall(r'\w{2,}', n)) for n in names)
    rec['doi_format_ok'] = (rec['doi'] == '') or bool(re.fullmatch(r'10\.\d{4,9}/\S+', rec['doi']))
    rec['type_ok'] = (rec['document_type'] == '') or rec['document_type'] in DOC_TYPES
    rows.append(rec)

extractions = pd.DataFrame(rows)
extractions.to_csv(OUT_DIR / 'eurollm_extractions.csv', index=False, encoding='utf-8-sig')
print('Parsed OK:', int(extractions.parse_ok.sum()), '/', len(extractions))
print('Filled fields:'); display((extractions[FIELDS] != '').sum().to_frame('documents'))
print('Grounded (of the documents where the field is filled):')
display(pd.DataFrame({f: [extractions.loc[extractions[f.replace('grounded_', '')] != '', f].mean()] for f in ['grounded_title', 'grounded_doi', 'grounded_authors']}).round(2))
display(extractions[['file_name', 'title', 'authors', 'doi', 'publication_year', 'document_type', 'language']].head(10))

## 6. Evaluate against the gold standard

Only gold cells that are filled in are scored. Title: similarity ≥ 0.90 counts as correct. Authors/keywords: set F1 (names compared as sorted tokens, so `Smith, J.` equals `J. Smith`). Everything else: exact match. Use `dev` while tuning; use `test` once at the end.

In [ ]:
EVAL_SPLIT = 'dev'
gold = pd.read_csv(GOLD_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
gold = gold[gold.split == EVAL_SPLIT].set_index('document_id')
pred = extractions.set_index('document_id')

def name_key(n): return ' '.join(sorted(re.findall(r'[a-z0-9]+', squash_space(n))))
def squash_space(s):
    s = unicodedata.normalize('NFKD', str(s)).lower()
    return ''.join(c for c in s if not unicodedata.combining(c))

def f1(g, p, key):
    g, p = {key(x) for x in g}, {key(x) for x in p}
    if not g and not p: return 1.0
    tp = len(g & p)
    if not tp: return 0.0
    pr, rc = tp / len(p), tp / len(g)
    return 2 * pr * rc / (pr + rc)

records = []
for doc_id, g in gold.iterrows():
    if doc_id not in pred.index:
        continue
    p = pred.loc[doc_id]
    for f in FIELDS:
        truth = g[f].strip()
        if truth == '':
            continue
        answer = p[f]
        if truth.upper() == 'NONE':
            score = float(answer == '')
        elif f == 'title':
            score = difflib.SequenceMatcher(None, squash(truth), squash(answer)).ratio()
        elif f == 'authors':
            score = f1(as_list(truth), as_list(answer), name_key)
        elif f == 'keywords':
            score = f1(as_list(truth), as_list(answer), squash)
        elif f == 'doi':
            score = float(clean_doi(truth) == clean_doi(answer))
        else:
            score = float(squash(truth) == squash(answer))
        ok = score >= 0.90 if f == 'title' else (score == 1.0)
        records.append({'file_name': g['file_name'], 'field': f, 'gold': truth, 'pred': answer})

results = pd.DataFrame(records)
if results.empty:
    print('Nothing to score yet: fill in gold_metadata.csv and run the extraction for this split.')
else:

    results.to_csv(OUT_DIR / f'evaluation_{EVAL_SPLIT}.csv', index=False, encoding='utf-8-sig')
    display(results)